In [0]:
%pip install beautifulsoup4 langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu langdetect
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import requests
from langdetect import detect
from langchain_community.chat_models import ChatDatabricks

# Load Secrets
config_path = "/Workspace/Users/cse230001083@iiti.ac.in/config.json"
with open(config_path, "r") as file:
    secrets = json.load(file)
api_key = secrets.get("MY_API_KEY")
print(api_key)
# The Heavyweight Choice: Llama 3.1 70B Instruct
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct", 
    max_tokens=1024,
    model_kwargs={
        "temperature": 0.1 # Crucial: Keep it low to ensure it stays factual and robotic
    }
)

# Translation Tools
SARVAM_LANG_MAP = {
    'hi': 'hi-IN', 'bn': 'bn-IN', 'kn': 'kn-IN', 'ml': 'ml-IN',
    'mr': 'mr-IN', 'or': 'or-IN', 'pa': 'pa-IN', 'ta': 'ta-IN',
    'te': 'te-IN', 'gu': 'gu-IN', 'en': 'en-IN'
}

def detect_language(text):
    try:
        return SARVAM_LANG_MAP.get(detect(text), 'en-IN')
    except:
        return 'en-IN'

def sarvam_translate(text, source_lang, target_lang, api_key="YOUR_SARVAM_API_KEY"):
    if source_lang == target_lang:
        return text
    url = "https://api.sarvam.ai/translate"
    headers = {"api-subscription-key": api_key, "Content-Type": "application/json"}
    payload = {
        "input": [text], "source_language_code": source_lang,
        "target_language_code": target_lang, "speaker_gender": "Male",
        "mode": "formal", "model": "sarvam-translate"
    }
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['translations'][0]['translated_text']
    except Exception as e:
        print(f"Translation API Failed: {e}")
        return text

sk_w4bre0gz_8B3vpmOgYGycslaz6l1xecwF


In [0]:
from pyspark.sql.functions import col, concat_ws, lit

# 1. THE COMPLETE FILE LIST
all_files = [
    {"file": "bns_sections.csv", "category": "BNS_Statutes", "type": "law"},
    {"file": "BNSS_sections_accurate.csv", "category": "BNSS_Statutes", "type": "law"},
    {"file": "bsa_sections.csv", "category": "BSA_Statutes", "type": "law"},
    {"file": "Shivam.csv", "category": "Shivam", "type": "case"},
    {"file": "pune_porsche_crash.csv", "category": "pune_porsche_crash", "type": "case"},
    {"file": "himachal_ndps_surat_singh.csv", "category": "himachal_ndps_surat_singh", "type": "case"},
    {"file": "russian_national_ndps_vildanov.csv", "category": "russian_national_ndps_vildanov", "type": "case"},
    {"file": "hashimpura_massacre.csv", "category": "hashimpura_massacre", "type": "case"},
    {"file": "priyadarshini_mattoo.csv", "category": "priyadarshini_mattoo", "type": "case"},
    {"file": "vijay_mallya.csv", "category": "vijay_mallya", "type": "case"},
    {"file": "vodafone_tax_dispute.csv", "category": "vodafone_tax_dispute", "type": "case"}
]

base_volume_path = "/Volumes/workspace/legal_data/raw_csv_uploads/"
table_name = "workspace.legal_data.legal_autopsy_raw_docs"

print("🧹 Clearing old data to prevent duplicates...")
spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} (file_path STRING, case_category STRING, document_text STRING) USING DELTA")
spark.sql(f"DELETE FROM {table_name}")

def ingest_file(file_info):
    full_path = f"{base_volume_path}{file_info['file']}"
    category = file_info['category']
    file_type = file_info.get('type', 'case') # Default to 'case' if type is missing
    
    try:
        # Load CSV
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("multiLine", "true") \
            .option("escape", '"') \
            .load(full_path)
        
        # CLEANUP: Trim invisible spaces from all column names
        for col_name in df.columns:
            df = df.withColumnRenamed(col_name, col_name.strip())
        
        # Re-check columns after cleaning
        clean_cols = df.columns
        
        if file_type == "law":
            # Statutes: Handle the BNS-specific logic
            # Use 'Section _name' or 'Section_name' depending on the file
            title_col = "Section _name" if "Section _name" in clean_cols else "Section_name"
            formatted_df = df.withColumn(
                "document_text",
                concat_ws(" - ", concat_ws(" ", lit("Section"), col("Section")), col(title_col), col("Description"))
            )
        else:
            # Cases: Look for 'document_text' OR 'Description'
            target_col = "document_text" if "document_text" in clean_cols else "Description"
            formatted_df = df.withColumn("document_text", col(target_col))
        
        # Final formatting
        final_df = formatted_df.withColumn("case_category", lit(category)) \
                               .withColumn("file_path", lit(full_path)) \
                               .select("file_path", "case_category", "document_text")
        
        # Write to table
        final_df.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"✅ Successfully loaded {final_df.count()} rows for: {category}")
        
    except Exception as e:
        print(f"❌ Error processing {file_info['file']}: {str(e)}")

print("🚀 Starting clean ingestion pipeline...")
for item in all_files:
    ingest_file(item)
print("🎉 All data loaded!")

🧹 Clearing old data to prevent duplicates...
🚀 Starting clean ingestion pipeline...
✅ Successfully loaded 358 rows for: BNS_Statutes
✅ Successfully loaded 506 rows for: BNSS_Statutes
✅ Successfully loaded 170 rows for: BSA_Statutes
✅ Successfully loaded 1 rows for: Shivam
✅ Successfully loaded 1 rows for: pune_porsche_crash
✅ Successfully loaded 1 rows for: himachal_ndps_surat_singh
✅ Successfully loaded 2 rows for: russian_national_ndps_vildanov
✅ Successfully loaded 1 rows for: hashimpura_massacre
✅ Successfully loaded 1 rows for: priyadarshini_mattoo
✅ Successfully loaded 1 rows for: vijay_mallya
✅ Successfully loaded 1 rows for: vodafone_tax_dispute
🎉 All data loaded!


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

df = spark.read.table("workspace.legal_data.legal_autopsy_raw_docs")
available_cases = [row['case_category'] for row in df.select('case_category').distinct().collect() if row['case_category'] is not None]

print(f"Available categories in database: {available_cases}")

rows = df.collect()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)
docs = []

for row in rows:
    text = row["document_text"]
    category = row["case_category"]
    if text:
        docs.append(Document(page_content=text, metadata={"case_category": category}))

chunks = text_splitter.split_documents(docs)
print(f"Boom. Sliced {len(rows)} documents into {len(chunks)} searchable chunks.")

print("Building Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)
print("Vectors successfully created and stored!")

Available categories in database: ['BNS_Statutes', 'BNSS_Statutes', 'vijay_mallya', 'BSA_Statutes', 'pune_porsche_crash', 'Shivam', 'hashimpura_massacre', 'priyadarshini_mattoo', 'himachal_ndps_surat_singh', 'vodafone_tax_dispute', 'russian_national_ndps_vildanov']


Boom. Sliced 1043 documents into 1557 searchable chunks.
Building Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectors successfully created and stored!


In [0]:
CASE_CATALOG = {
    "Shivam": """
        Themes: Unnao rape case, sexual assault, victim intimidation, 
        blackmail with videos/digital recordings, kidnapping, political corruption, 
        police negligence, custodial failures, and murder of witnesses/victims. 
        Keywords: Rae Bareili, Unnao, blackmail, video evidence.
    """,
    "pune_porsche_crash": """
        Themes: Underage/Minor driving, Drunk Driving (DUI), road safety, 
        fatal car accidents, Juvenile Justice Act, JJB bail conditions (essay writing), 
        Culpable Homicide, Negligence, Motor Vehicles Act, 
        blood sample tampering at Sassoon Hospital, parental liability, and pub/bar owner liability.
        Keywords: Vedant Agarwal, Porsche, essay, luxury car, minor arrest.
    """,
    ### need files for these
    "Md. Bani Alam Mazid (2025) 21-Year Death Row Acquittal": """
       Themes: Death penalty, long-term incarceration (21 years), investigative deficiencies, 
       procedural adherence lapses, standard of proof in capital crimes, appellate scrutiny.
       Keywords: Bani Alam Mazid, UP Police, death row, reasonable doubt. 
    """,
    "Best Bakery Case witnesses turn hostile": """
       Themes: witness intimidation, hostile witnesses, retrial, miscarriage of justice.
       Keywords: Best Bakery, Zahira Sheikh, Gujarat riots, witness protection failure.
    """,
    "Nitish Katara murder case":  """
        Themes: delayed justice, political influence, honour killing, prolonged litigation.
        Keywords: Nitish Katara, honour killing, trial delay, influence, conspiracy laws.
    """,
    "Nithari killings": """
        Themes: serial killings, forensic gaps, death penalty reversal, insufficient evidence.
        Keywords: Nithari, Moninder Pandher, Surinder Koli, acquittal, death penalty jurisprudence.
    """,
    ###need file for these
    "himachal_ndps_surat_singh": """
        Themes: Narcotics (NDPS Act), procedural search loopholes, 
        Section 50 NDPS non-compliance (third option error), IO investigative lapses, 
        forensic equipment failure (traditional vs electronic scales), 
        and technical acquittals of high-volume traffickers.
        Keywords: Surat Singh, Himachal, contraband weighing, invalid consent, procedural acquittal.
    """,

    "russian_national_ndps_vildanov": """
        Themes: Smuggling, foreign national rights, investigative sequence manipulation, 
        pre-signed consent letters, border security lapses (Indo-Nepal), 
        confession timing discrepancies, and failure to prove guilt beyond reasonable doubt.
        Keywords: Doniyar Vildanov, Russian national, charas, border arrest, mandatory prescription.
    """,

    "hashimpura_massacre": """
        Themes: Extrajudicial custodial killings, communal violence (PAC misconduct), 
        systemic evidence suppression (ACR omissions), decades-long judicial delay, 
        acquittal reversal due to RTI evidence, and the failure of state accountability.
        Keywords: Meerut 1987, PAC, life imprisonment, police canal bodies, RTI discovery.
    """,

    "priyadarshini_mattoo": """
        Themes: Rape and murder, influence of high-ranking officials (IG's son), 
        forensic/DNA evidence scrutiny, trial court acquittal vs HC reversal, 
        judicial shift from death penalty to life (commutation), and stalking harassment.
        Keywords: Santosh Singh, IG son, Delhi 1996, DNA report doubt, life imprisonment.
    """,

    "vijay_mallya": """
        Themes: Financial fraud, white-collar crime leniency, money laundering, 
        extradition delays (UK-India), Fugitive Economic Offenders Act, 
        nominal sentencing for contempt (4 months), and systemic debt recovery friction.
        Keywords: Kingfisher, extradition, contempt of court, US$40 million, fugitive.
    """,

    "vodafone_tax_dispute": """
        Themes: International taxation, corporate legal advantage, 
        'Corporate Guarantee' vs 'Cash Deposit' discrepancy, judicial disconnect 
        favoring MNCs over state revenue, and the complexity of offshore valuations.
        Keywords: Vodafone, Bombay HC, ₹1100-crore tax, corporate guarantee, ITAT.
    """

}

In [0]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
[SCENARIO: You are an elite Legal Analyst performing a "Legal Autopsy" on a specific case. 
Your objective is to answer the user's question by explaining HOW and WHY certain legal events 
happened using ONLY the provided case facts and statutory laws. You must be brutally honest, 
clinically objective, and forensic in your identification of systemic failure.]

### ABSOLUTE RULES (ANTI-HALLUCINATION & ANTI-BLUFFING):
1. THE IPC BAN & HISTORICAL TRANSLATION: The [CASE FACT] blocks might mention old IPC/CrPC 
   sections. You must act as a legal translator: identify the crime in the facts, and verify 
   it using ONLY the new BNS, BNSS, and BSA sections provided. NEVER cite the IPC or CrPC 
   in your "Statutory Verification" or "Autopsy" sections.
2. EXACT DEFINITIONS: Use only the exact text provided in [STATUTE] blocks.
3. STRICT TRANSPARENCY: If a context says "[ERROR: NO RELEVANT...]", you MUST explicitly 
   state that the database lacks this information. Do not improvise.
4. NO ENDLESS LISTS: Summarize the core intent of long statutes in plain English.
5. FACTUAL CRITIQUE: Rely ONLY on discrepancies between the statutory ideal and the 
   documented case facts. Avoid sensational or moralizing language.
6. STRICT SOURCING: Every claim must be traceable to a specific [STATUTE] or [CASE FACT].
7. IDENTIFY THE ACTOR: When discussing flaws, specify which entity (Police, Magistrate, 
   Judicial Board, or Prosecution) is responsible for the lapse.

### YOUR INSTRUCTIONS:
Structure your response exactly like this:

**1. The Facts of the Matter**
Briefly state what actually happened in the case, citing the [CASE FACT] blocks. (Mention 
old IPC numbers here ONLY if quoting the historical FIR facts).

**2. Statutory Verification (Isolated)**
* **BNSS (Procedure):** Quote the exact new procedural law found. If ERROR, output: "Transparency Note: No procedural laws (BNSS) were found."
* **BNS (Crimes):** Quote the exact new substantive law found. If ERROR, output: "Transparency Note: No substantive criminal laws (BNS) were found."
* **BSA (Evidence):** Quote the exact new evidentiary law found. If ERROR, output: "Transparency Note: No evidentiary laws (BSA) were found."

**3. Expert Legal Autopsy**
Combine the facts and the NEW statutes to answer the question. Explain how the historical 
charges should be interpreted under the modern BNS/BNSS framework.

**4. Investigation & Police Lapses (Friction)**
Identify where the Executive/Police branch failed. Compare the [CASE FACT] to the [STATUTE] 
requirements for investigation, arrest, and evidence collection. Highlight where protocol 
was bypassed.

**5. Judicial Disconnect & The Justice Gap**
Analyze the Judicial outcome (Bail, Sentencing, or Orders). Identify the "Justice Gap"—the 
discrepancy between the severity of the BNS/BNSS statutory requirement and the actual 
Judicial decision. Point out where the court’s interpretation appears to contradict the 
legislative intent of the new laws.

--------------------------------------------------
CONTEXT:
{context}

ROUTED CASE SUMMARY:
{case_summary}

USER QUESTION: {question}

EXPERT LEGAL AUTOPSY:
"""

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question", "case_summary"])

In [0]:
def route_query(user_query, case_catalog, llm_model):
    catalog_string = "\n".join([f"- {name}: {desc}" for name, desc in case_catalog.items()])
    router_template = """
    You are an intelligent routing assistant for a legal database.
    Your job is to read the User Question and match it to the most relevant legal case from the Case Catalog.
    
    Case Catalog:
    {catalog}
    
    User Question: "{query}"
    
    Rules:
    1. Reply with EXACTLY ONE case name from the catalog.
    2. Do NOT add any extra text, punctuation, or explanation.
    3. If the question doesn't match any case's description, reply with the word: ALL
    
    Category Match:"""
    
    router_prompt = PromptTemplate(template=router_template, input_variables=["catalog", "query"])
    prompt_text = router_prompt.format(catalog=catalog_string, query=user_query)
    decision = llm_model.invoke(prompt_text).content.strip()
    
    valid_cases = list(case_catalog.keys())
    if decision not in valid_cases and decision != "ALL":
        return "ALL"
    return decision

In [0]:
def generate_multilingual_legal_response(user_query, vector_store, llm_model, prompt_template, case_catalog):
    user_lang_code = detect_language(user_query)
    english_query = sarvam_translate(user_query, source_lang=user_lang_code, target_lang="en-IN")
    
    target_category = route_query(english_query, case_catalog, llm_model)
    print(f"[System] User intent matched to real-world case: {target_category}")
    
    # PHASE 1: ISOLATED CASE FACT RETRIEVAL
    case_context = "[ERROR: NO CASE FACTS FOUND FOR THIS QUERY.]"
    if target_category != "ALL" and target_category in case_catalog:
        print(f"[Debug] Fetching Case Facts for: {target_category}...")
        case_docs = vector_store.similarity_search(english_query, k=8, filter={"case_category": target_category})
        if case_docs:
            case_context = "\n".join([f"- [CASE FACT]: {d.page_content}" for d in case_docs])
    
    # PHASE 2: ISOLATED STATUTE RETRIEVAL (One by One)
    print("[Debug] Fetching Statutory Verification (Isolated)...")
    
    bnss_docs = vector_store.similarity_search(english_query, k=5, filter={"case_category": "BNSS_Statutes"})
    bnss_context = "\n".join([f"- [STATUTE]: {d.page_content}" for d in bnss_docs]) if bnss_docs else "[ERROR: NO RELEVANT BNSS (PROCEDURAL) LAWS FOUND IN DATABASE.]"
    
    bns_docs = vector_store.similarity_search(english_query, k=5, filter={"case_category": "BNS_Statutes"})
    bns_context = "\n".join([f"- [STATUTE]: {d.page_content}" for d in bns_docs]) if bns_docs else "[ERROR: NO RELEVANT BNS (CRIMINAL) LAWS FOUND IN DATABASE.]"
    
    bsa_docs = vector_store.similarity_search(english_query, k=5, filter={"case_category": "BSA_Statutes"})
    bsa_context = "\n".join([f"- [STATUTE]: {d.page_content}" for d in bsa_docs]) if bsa_docs else "[ERROR: NO RELEVANT BSA (EVIDENCE) LAWS FOUND IN DATABASE.]"

    strict_context = f"""
    === CASE FACTS ({target_category}) ===
    {case_context}
    === BNSS VERIFICATION (Procedure) ===
    {bnss_context}
    === BNS VERIFICATION (Crimes/Punishments) ===
    {bns_context}
    === BSA VERIFICATION (Evidence) ===
    {bsa_context}
    """
    # Add this inside your generation function to see what's happening
    print(f"DEBUG: Case category detected: {target_category}")
    print(f"DEBUG: Number of case facts retrieved: {len(case_docs)}")
    
    fallback_summary = case_catalog.get(target_category, "General Indian Legal Framework")
    prompt_text = prompt_template.format(context=strict_context, question=english_query, case_summary=fallback_summary)
    
    english_response = llm_model.invoke(prompt_text).content
    final_response = sarvam_translate(english_response, source_lang="en-IN", target_lang=user_lang_code)
    
    return {
        "detected_language": user_lang_code,
        "routed_category": target_category,
        "final_response": final_response
    }

In [0]:
# 1. Create input box
dbutils.widgets.text("query", "Ask your legal question")
query = dbutils.widgets.get("query")

# 2. Run only if user entered something
if query.strip() != "" and query.strip() != "Ask your legal question":
    try:
        result = generate_multilingual_legal_response(
            user_query=query,
            vector_store=vector_db,
            llm_model=llm,
            prompt_template=prompt,
            case_catalog=CASE_CATALOG
        )
        answer = result["final_response"]
    except Exception as e:
        answer = f"Error: {str(e)}"
else:
    answer = "Please enter a legal question."

# 3. Display UI
displayHTML(f"""
<div style="font-family: Arial; padding: 20px; border-radius: 12px; background-color: #f5f5f5;">
    <h2>⚖️ AI Legal Autopsy</h2>
    <p><b>🔍 Question:</b><br>{query}</p>
    <hr>
    <p><b>📜 Autopsy Report:</b><br>{answer}</p>
</div>
""")

[System] User intent matched to real-world case: vodafone_tax_dispute
[Debug] Fetching Case Facts for: vodafone_tax_dispute...
[Debug] Fetching Statutory Verification (Isolated)...
DEBUG: Case category detected: vodafone_tax_dispute
DEBUG: Number of case facts retrieved: 3


⚖️ AI Legal Autopsy 
 🔍 Question: "The Bombay High Court allowed Vodafone to provide a 'corporate guarantee' instead of a cash deposit for a 1100-crore tax demand. Is this a Judicial Disconnect that favors multinational corporations over the state's revenue? Contrast this with how the law handles tax recovery for smaller, local entities." 
 
 📜 Autopsy Report: **1. The Facts of the Matter**
The Bombay High Court upheld the Income Tax Appellate Tribunal's (ITAT) order, which directed Vodafone India Services Pvt. Ltd. to pay ₹230 crores, a significantly reduced amount from the original recovery amount of ₹1128.46 crores. The Court also modified the condition imposed by the ITAT, allowing Vodafone to furnish a corporate guarantee from its ultimate parent, Vodafone International Holdings BV, Netherlands, instead of a corporate guarantee from an associate company with unencumbered assets in India.

**2. Statutory Verification (Isolated)**
* **BNSS (Procedure):** The relevant procedural law is found in Section 325 of the Bharatiya Nyaya Sanhita, 2023, which deals with the execution of foreign commissions. Additionally, Section 384 of the Bharatiya Nyaya Sanhita, 2023, provides the procedure for contempt cases. Section 65 of the Bharatiya Nyaya Sanhita, 2023, outlines the service of summons on corporate bodies, firms, and societies.
* **BNS (Crimes):** Transparency Note: No substantive criminal laws (BNS) were found.
* **BSA (Evidence):** The relevant evidentiary law is found in Section 88 of the Bharatiya Nyaya Sanhita, 2023, which deals with the presumption of certified copies of foreign judicial records.

**3. Expert Legal Autopsy**
The Bombay High Court's decision to allow Vodafone to provide a corporate guarantee instead of a cash deposit for the ₹1100-crore tax demand raises questions about the judicial approach to tax recovery. Under the Bharatiya Nyaya Sanhita, 2023, the Court has the discretion to modify conditions imposed by the ITAT. However, the decision seems to favor Vodafone, a multinational corporation, over the state's revenue. The use of a corporate guarantee instead of a cash deposit may be seen as a more lenient approach, potentially creating a disparity in the treatment of smaller, local entities.

**4. Investigation & Police Lapses (Friction)**
In this case, the investigation and police lapses are not directly relevant, as the matter pertains to tax recovery and judicial proceedings. However, the ITAT's decision to impose conditions on Vodafone, which were later modified by the High Court, highlights the importance of careful consideration in tax recovery cases. The Executive/Police branch is not directly involved in this matter, but the ITAT's and the High Court's decisions demonstrate the need for a balanced approach to tax recovery, ensuring that the state's revenue is protected while also considering the interests of taxpayers.

**5. Judicial Disconnect & The Justice Gap**
The Judicial outcome in this case raises concerns about a potential judicial disconnect, favoring multinational corporations over the state's revenue. The decision to allow Vodafone to provide a corporate guarantee instead of a cash deposit may be seen as a more lenient approach, potentially creating a disparity in the treatment of smaller, local entities. The Court's interpretation of the law, although within its discretion, may be perceived as contradictory to the legislative intent of ensuring fair and efficient tax recovery. The justice gap in this case lies in the potential disparity in treatment between large corporations and smaller entities, highlighting the need for a more balanced and consistent approach to tax recovery.